<a href="https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [3]:
df = pd.read_csv('/content/drive/MyDrive/merged_and_cleaned_dim_content_and_fact_content_daily_performance_month_2026_05.csv')
df.columns

Index(['content_hash_id', 'client', 'impressions', 'clicks', 'sum_position',
       'days_with_gsc', 'avg_position', 'ctr', 'client_hash_id',
       'keyword_hash_id', 'url_hash_id', 'keyword_char_count',
       'keyword_token_count', 'url_char_count', 'content_created_date',
       'content_updated_date', 'content_type', 'search_volume', 'competition',
       'competition_level', 'cpc', 'main_intent', 'backlinks',
       'category_count', 'keyword_created_date', 'char_count', 'word_count',
       'is_published', 'is_deleted', '_merge'],
      dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134279 entries, 0 to 134278
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   content_hash_id       134279 non-null  object 
 1   client                134279 non-null  object 
 2   impressions           134279 non-null  int64  
 3   clicks                134279 non-null  int64  
 4   sum_position          134279 non-null  int64  
 5   days_with_gsc         134279 non-null  int64  
 6   avg_position          134279 non-null  float64
 7   ctr                   134279 non-null  float64
 8   client_hash_id        134279 non-null  object 
 9   keyword_hash_id       134279 non-null  object 
 10  url_hash_id           134276 non-null  object 
 11  keyword_char_count    134279 non-null  int64  
 12  keyword_token_count   134279 non-null  int64  
 13  url_char_count        134279 non-null  int64  
 14  content_created_date  134279 non-null  object 
 15  

In [5]:
earliest_date_of_creation = df['content_created_date'].min()
latest_date_of_creation = df['content_created_date'].max()

earliest_date_of_update = df['content_updated_date'].min()
latest_date_of_upate = df['content_updated_date'].max()

In [6]:
df['old'] = np.where(df['content_created_date'] <= '2026-03-15' , 1, 0)
conditions = (df['old']==1) & (df['content_updated_date'] <='2026-06-13')
df['old_but_updated_recently'] = np.where(conditions, 0, 1)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method:

I use KMeans on nine structural features (word_count, char_count, keyword_token_count, url_char_count, category_count, backlinks, search_volume, competition, cpc; log-transformed and standardised on train). K=5 was chosen from structure-only criteria, before any CTR comparison: among K ≥ 3 it has the highest silhouette (0.247), stability 0.998 across seeds and no cluster below 8.6% of training pages. K=2 scores higher on silhouette and stability but yields only two broad groups. Sensitivity to K and feature set is reported in section 3.


 I did not use Random Forest or Gradient Boosting: they add complexity without answering whether archetypes carry signal. k=3 was chosen on structure-only criteria (no tiny clusters, stable across seeds) before any CTR results were seen.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: I split at the page level by content_created_date: older pages train, newer pages test, with a check that no content_hash_id appears on both sides. This matches the real use of the benchmark, which will be applied to content newer than the content it learned from, and it guarantees the model never sees a test page during training. All learned parts (scaler, clusters, benchmarks) are fit on the training pages only.

Limits. The outcome metrics come from the same report window for both groups, so this is an older-vs-newer split, not a forecast, and age differences between the groups may contribute to any gap. Clients are not held out: 20 clients appear in both train and test, 11 appear only in test and 8 only in train, so the split is partly new-client. I report errors per client to check the result is not driven by one client. The Week-4 eligibility filter (old, not recently updated) matches no test page, so the ranking is compared on all test pages.

In [8]:
df['content_created_date'].describe()

,content_created_date
count,134279
unique,221
top,2026-04-23
freq,4406


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

d = df.copy()
d['pos_x_imp'] = d['avg_position'] * d['impressions']

In [10]:
struct_cols = ['client', 'content_type', 'word_count', 'char_count',
               'keyword_token_count', 'url_char_count', 'category_count', 'search_volume', 'competition', 'cpc']
check = d.groupby('content_hash_id')[['content_created_date'] + struct_cols].nunique()
print("pages with >1 value per column (should all be 0):")
print((check > 1).sum())

pages with >1 value per column (should all be 0):
content_created_date    0
client                  0
content_type            0
word_count              0
char_count              0
keyword_token_count     0
url_char_count          0
category_count          0
search_volume           0
competition             0
cpc                     0
dtype: int64


In [11]:
pages = (d.sort_values('days_with_gsc')).groupby('content_hash_id').agg(client=('client', 'first'),
                content_type=('content_type', 'first'),
                main_intent=('main_intent', 'first'),
                created=('content_created_date', 'min'),
                word_count=('word_count', 'first'),
                char_count=('char_count', 'first'),
                keyword_token_count=('keyword_token_count', 'first'),
                url_char_count=('url_char_count', 'first'),
                category_count=('category_count', 'first'),
                backlinks=('backlinks', 'first'),
                search_volume=('search_volume', 'first'),
                competition=('competition', 'first'),
                cpc=('cpc', 'first'),
                impressions=('impressions', 'sum'),
                clicks=('clicks', 'sum'),
                old=('old', 'last'),
                old_but_updated_recently=('old_but_updated_recently', 'last'),
                pos_x_imp=('pos_x_imp', 'sum'),
                n_reports=('days_with_gsc', 'nunique')).reset_index()

In [12]:
pages['avg_position'] = pages['pos_x_imp'] / pages['impressions']
pages['ctr'] = pages['clicks'] / pages['impressions']
pages = pages[pages['impressions'] >= 100].copy()
pages['pos_bucket'] = pd.cut(pages['avg_position'], [0, 3, 5, 7, 10, 20, 50, 100])

print("pages:", len(pages), "| unique ids:", pages['content_hash_id'].nunique())

pages: 70709 | unique ids: 70709


In [13]:
pages['created'] = pd.to_datetime(pages['created'], errors='coerce')

In [14]:
print(pages['created'].describe())
for q in [0.6, 0.7, 0.8]:
    print(f"{q:.0%} of pages were created before", pages['created'].quantile(q).date())

count                            70709
mean     2026-01-27 03:15:09.558896384
min                2024-12-19 00:00:00
25%                2025-12-12 00:00:00
50%                2026-02-17 00:00:00
75%                2026-04-02 00:00:00
max                2026-06-05 00:00:00
Name: created, dtype: object
60% of pages were created before 2026-03-05
70% of pages were created before 2026-03-23
80% of pages were created before 2026-04-13


In [15]:
CUTOFF = pd.Timestamp('2026-03-21')
train = pages[pages['created'] < CUTOFF].copy()
test  = pages[pages['created'] >= CUTOFF].copy()

In [16]:
overlap = set(train['content_hash_id']) & set(test['content_hash_id'])
print("\npage overlap between train and test:", len(overlap))
assert len(overlap) == 0


page overlap between train and test: 0


In [17]:
print("pages      train/test:", len(train), len(test))
print("impressions train/test:", train['impressions'].sum(), test['impressions'].sum())
print("test share of pages: %.1f%%" % (100 * len(test) / len(pages)))

pages      train/test: 49232 21477
impressions train/test: 137736188 32267376
test share of pages: 30.4%


In [18]:
print("\nclients in train:", train['client'].nunique(), "| in test:", test['client'].nunique())
print("clients appearing in both:", len(set(train['client']) & set(test['client'])))
print("\ntest pages per client (top 5):")
print(test['client'].value_counts().head())


clients in train: 28 | in test: 31
clients appearing in both: 20

test pages per client (top 5):
client
client_8ddc46da5414ffd8    3367
client_73cda7b4e4f265ea    2086
client_e5c2aa26a8598242    1780
client_62f4a7e64f5e0096    1686
client_86ebc2f12c01f586    1647
Name: count, dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [19]:
pages.head(10)

,content_hash_id,client,content_type,main_intent,created,word_count,char_count,keyword_token_count,url_char_count,category_count,...,cpc,impressions,clicks,old,old_but_updated_recently,pos_x_imp,n_reports,avg_position,ctr,pos_bucket
0,content_00000c99413ae2ad,client_7de9989c909e91a5,keyword article,informational,2026-05-16,2446.0,15660.0,4,105,3,...,0.88,112,2,0,1,854.0,1,7.625000,0.017857,"(7, 10]"
1,content_000217b3de34df5a,client_e5c2aa26a8598242,keyword article,informational,2026-05-12,3038.0,17897.0,6,100,0,...,0.00,1743,5,0,1,85435.0,1,49.016064,0.002869,"(20, 50]"
3,content_00022fd55ac280be,client_3f0ce4d44fe94f3d,keyword article,transactional,2025-12-16,2785.0,18216.0,7,90,3,...,0.00,3227,2,1,0,63985.0,1,19.828014,0.000620,"(10, 20]"
4,content_00032be2df0005ca,client_fef1a8f436438636,keyword article,commercial,2026-02-05,1502.0,9441.0,9,113,4,...,0.00,218,1,1,1,3882.0,1,17.807339,0.004587,"(10, 20]"
5,content_00039f4c7a954114,client_157ffe4d4a595515,comparison article,informational,2026-01-09,2573.0,17104.0,9,125,0,...,0.00,110,0,1,0,1118.0,1,10.163636,0.000000,"(10, 20]"
11,content_000612ace4167db9,client_73cda7b4e4f265ea,keyword article,commercial,2025-09-26,2249.0,14576.0,9,121,6,...,0.14,1835,0,1,0,48881.0,1,26.638147,0.000000,"(20, 50]"
13,content_0008f4a4b35f399b,client_e547b89c05043229,keyword article,transactional,2025-09-05,1627.0,9981.0,7,87,8,...,0.00,433,2,1,0,14471.0,1,33.420323,0.004619,"(20, 50]"
15,content_0009df3c2b96f209,client_73cda7b4e4f265ea,keyword article,informational,2026-02-03,2999.0,20442.0,5,79,6,...,0.00,126,0,1,0,1469.0,1,11.658730,0.000000,"(10, 20]"
16,content_000a3a10f7714a23,client_73cda7b4e4f265ea,keyword article,informational,2026-04-13,2697.0,17986.0,9,88,0,...,0.00,3392,50,0,1,22149.0,1,6.529776,0.014741,"(5, 7]"
18,content_000b09c2a4ef4ba8,client_e5c2aa26a8598242,keyword article,informational,2026-02-10,2840.0,18350.0,8,105,7,...,0.07,255,0,1,0,11878.0,1,46.580392,0.000000,"(20, 50]"


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
CORE = ['word_count','char_count','keyword_token_count','url_char_count',
       'category_count','backlinks','search_volume','competition','cpc']

In [21]:
def make_X(df):
  X = np.log1p(df[CORE].clip(lower=0))
  return X

In [22]:
train_x = make_X(train)
test_x = make_X(test).reindex(columns=train_x.columns, fill_value=0.0)

In [23]:
sc = StandardScaler().fit(train_x)
scaled_x_train , scaled_x_test = sc.transform(train_x), sc.transform(test_x)

In [24]:
rng = np.random.RandomState(0)
sub = rng.choice(len(scaled_x_train), size=min(8000, len(scaled_x_train)), replace=False)
rows = []
for k in range(2, 9):
  labs = [KMeans(k, n_init=5, random_state=s).fit(scaled_x_train).labels_ for s in  (1, 2, 3)]
  rows.append(dict(k=k,
                     silhouette=silhouette_score(scaled_x_train[sub], labs[0][sub]),
                     stability_ari=np.mean([adjusted_rand_score(labs[0], labs[j]) for j in (1, 2)]),
                     smallest_cluster=np.bincount(labs[0]).min()))
print(pd.DataFrame(rows).round(3))

   k  silhouette  stability_ari  smallest_cluster
0  2       0.288          1.000             20241
1  3       0.205          0.644             13117
2  4       0.239          0.999              5651
3  5       0.247          0.998              4258
4  6       0.233          0.999              4228
5  7       0.236          0.908              2791
6  8       0.241          0.846              2444


In [25]:
K = 5
km = KMeans(K, n_init=10, random_state=42).fit(scaled_x_train)
train['cluster'] = km.labels_
test['cluster']  = km.predict(scaled_x_test)

In [26]:
prof = train.groupby('cluster').agg(
    pages=('ctr','size'), words=('word_count','median'), backlinks=('backlinks','median'),
    volume=('search_volume','median'), cpc=('cpc','median'),
    top_intent=('main_intent', lambda s: s.mode().iat[0]),
    avg_pos=('avg_position','median'),
    impressions=('impressions','median'),
    weighted_ctr=('clicks', 'sum'))
prof['weighted_ctr'] = prof['weighted_ctr'] / train.groupby('cluster')['impressions'].sum()
print(prof.round(4))

         pages   words  backlinks  volume   cpc     top_intent  avg_pos  \
cluster                                                                   
0        14986  2715.0        0.0     0.0  0.00  informational  11.3018   
1         4258  2709.0       11.0    30.0  2.23  transactional  14.0743   
2        13080  3902.0        0.0     0.0  0.00  informational  23.4054   
3        10177  2676.0        0.0    10.0  0.00  informational  12.0124   
4         6731  2725.0      172.0    40.0  0.00  informational  11.6571   

         impressions  weighted_ctr  
cluster                             
0              796.0        0.0032  
1              487.0        0.0033  
2              835.0        0.0038  
3              764.0        0.0034  
4              718.0        0.0032  


In [27]:
train.head()

,content_hash_id,client,content_type,main_intent,created,word_count,char_count,keyword_token_count,url_char_count,category_count,...,impressions,clicks,old,old_but_updated_recently,pos_x_imp,n_reports,avg_position,ctr,pos_bucket,cluster
3,content_00022fd55ac280be,client_3f0ce4d44fe94f3d,keyword article,transactional,2025-12-16,2785.0,18216.0,7,90,3,...,3227,2,1,0,63985.0,1,19.828014,0.000620,"(10, 20]",3
4,content_00032be2df0005ca,client_fef1a8f436438636,keyword article,commercial,2026-02-05,1502.0,9441.0,9,113,4,...,218,1,1,1,3882.0,1,17.807339,0.004587,"(10, 20]",3
5,content_00039f4c7a954114,client_157ffe4d4a595515,comparison article,informational,2026-01-09,2573.0,17104.0,9,125,0,...,110,0,1,0,1118.0,1,10.163636,0.000000,"(10, 20]",0
11,content_000612ace4167db9,client_73cda7b4e4f265ea,keyword article,commercial,2025-09-26,2249.0,14576.0,9,121,6,...,1835,0,1,0,48881.0,1,26.638147,0.000000,"(20, 50]",4
13,content_0008f4a4b35f399b,client_e547b89c05043229,keyword article,transactional,2025-09-05,1627.0,9981.0,7,87,8,...,433,2,1,0,14471.0,1,33.420323,0.004619,"(20, 50]",3


In [28]:
prof

,pages,words,backlinks,volume,cpc,top_intent,avg_pos,impressions,weighted_ctr
cluster,,,,,,,,,
0,14986,2715.0,0.0,0.0,0.00,informational,11.301836,796.0,0.003216
1,4258,2709.0,11.0,30.0,2.23,transactional,14.074328,487.0,0.003273
2,13080,3902.0,0.0,0.0,0.00,informational,23.405394,835.0,0.003776
3,10177,2676.0,0.0,10.0,0.00,informational,12.012397,764.0,0.003402
4,6731,2725.0,172.0,40.0,0.00,informational,11.657143,718.0,0.003249


In [29]:
for df in (train, test):
  df['pb'] = df['pos_bucket'].astype(str)

In [30]:
#position (Week 4 Baseline Rule)
g = train.groupby('pb')[['clicks', 'impressions']].sum()
mapA = g['clicks']/g['impressions']
test['predA'] = test['pb'].map(mapA)

In [31]:
#position x archetype, with fallback to A when a cell is too small
MIN_N = 50
cell = train.groupby(['pb', 'cluster']).agg(n=('ctr', 'size'), clicks=('clicks', 'sum'), imps=('impressions', 'sum'))
cell['pred'] = cell['clicks'] / cell['imps']
cell = cell.loc[cell['n'] >= MIN_N, 'pred']
test = test.join(cell.rename('predB'), on=['pb','cluster'])
print("share of test pages using the fallback in B: %.1f%%" % (100 * test['predB'].isna().mean()))
test['predB'] = test['predB'].fillna(test['predA'])

share of test pages using the fallback in B: 0.1%


In [32]:
Xtr = pd.DataFrame(scaled_x_train, columns=train_x.columns)
Xte = pd.DataFrame(scaled_x_test, columns=train_x.columns)

In [33]:
#shallow decision tree on position + structure (no cluster label)
from sklearn.tree import DecisionTreeRegressor
XtrC = Xtr.assign(avg_position=train['avg_position'])
XteC = Xte.assign(avg_position=test['avg_position'])
tree = DecisionTreeRegressor(max_depth=4, min_samples_leaf=100, random_state=0)
tree.fit(XtrC, train['ctr'], sample_weight=train['impressions'])
test['predC'] = tree.predict(XteC)

In [34]:
def wmae(y, p, w): return np.average(np.abs(y - p), weights=w)

y, w = test['ctr'].values, test['impressions'].values
res = pd.DataFrame({m: [wmae(y, test['pred'+m].values, w)] for m in 'ABC'}, index=['weighted_MAE']).T
res['vs_A_%'] = 100 * (res.loc['A','weighted_MAE'] - res['weighted_MAE']) / res.loc['A','weighted_MAE']
print(res.round(6))

def boot_diff(pa, px, n=1000, seed=0):
    r = np.random.default_rng(seed); ea, ex = np.abs(y-pa), np.abs(y-px); out = []
    for _ in range(n):
        s = r.integers(0, len(y), len(y))
        out.append(np.average(ex[s]-ea[s], weights=w[s]))
    return np.percentile(out, [2.5, 50, 97.5])

for m in 'BC':
    lo, mid, hi = boot_diff(test['predA'].values, test['pred'+m].values)
    print(f"{m} minus A: median {mid:.6f}, 95% interval [{lo:.6f}, {hi:.6f}]")

   weighted_MAE    vs_A_%
A      0.003445  0.000000
B      0.003533 -2.555542
C      0.003616 -4.968835
B minus A: median 0.000086, 95% interval [0.000048, 0.000140]
C minus A: median 0.000174, 95% interval [0.000101, 0.000238]


On 21,477 held-out pages, position improves on a constant benchmark by about 4% (MAE 0.003445 vs 0.003584) and 3% (RMSE). The archetype-aware benchmark is measurably worse than position alone (MAE +2.6%, RMSE +1.6%; interval for B minus A [+0.000048, +0.000140]), and so is the tree (MAE +5.0%). The simpler position-only benchmark is kept. [Add the sensitivity table result here.]

In [35]:
# ranking check: same eligibility filter as the Week-4 rule
elig = (test['old'] == 1) & (test['old_but_updated_recently'] == 0)
top = {}
for m in 'ABC':
    test['miss'+m] = (test['impressions'] * (test['pred'+m] - test['ctr'])).clip(lower=0)
    top[m] = set(test[elig].nlargest(20, 'miss'+m)['content_hash_id'])
print("top-20 overlap  A&B:", len(top['A'] & top['B']), " A&C:", len(top['A'] & top['C']), " B&C:", len(top['B'] & top['C']))

top-20 overlap  A&B: 0  A&C: 0  B&C: 0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Errors. The archetype-aware benchmark has higher error than the baseline in all five clusters (+1% to +7%, largest in cluster 4) and in six of seven position buckets. Zero-click pages show the opposite direction (0.00264 vs 0.00281) but are a minority of impressions. All benchmarks under-predict the test pages (bias about −0.0017 for A, roughly half its MAE), consistent with a level shift between older and newer pages; I did not isolate the cause. The mix of archetypes also changed (cluster 2 falls from 27% of training pages to 9% of test pages). Errors vary more across clients (MAE 0.0020 to 0.0045 among the five largest test clients) than between benchmarks (at most 0.0002 within a client), so client-level differences are a directional explanation I did not test. Permutation importance is near zero for every feature, so the tree has no features it relies on in a way that transfers to the test set. Conclusion. In this sample, adding content archetype to the expected-CTR benchmark did not improve prediction on held-out newer pages, and a shallow tree did worse; the position-only baseline stays as the decision-support benchmark.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# where does each benchmark go wrong?
def slice_err(col):
    return test.groupby(col).apply(lambda d: pd.Series({
        'pages': len(d), 'MAE_A': wmae(d['ctr'], d['predA'], d['impressions']),
        'MAE_B': wmae(d['ctr'], d['predB'], d['impressions'])}))
print(slice_err('cluster').round(5))
print(slice_err('client').sort_values('pages', ascending=False).head(5).round(5))

          pages    MAE_A    MAE_B
cluster                          
0        8750.0  0.00333  0.00337
1        1876.0  0.00404  0.00415
2        2014.0  0.00315  0.00321
3        5772.0  0.00371  0.00378
4        3065.0  0.00319  0.00341
                          pages    MAE_A    MAE_B
client                                           
client_8ddc46da5414ffd8  3367.0  0.00454  0.00477
client_73cda7b4e4f265ea  2086.0  0.00199  0.00195
client_e5c2aa26a8598242  1780.0  0.00273  0.00282
client_62f4a7e64f5e0096  1686.0  0.00308  0.00308
client_86ebc2f12c01f586  1647.0  0.00243  0.00251


/tmp/ipykernel_12699/2062954912.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return test.groupby(col).apply(lambda d: pd.Series({
/tmp/ipykernel_12699/2062954912.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return test.groupby(col).apply(lambda d: pd.Series({


In [37]:
y, w = test['ctr'].values, test['impressions'].values
test['predK'] = train['clicks'].sum() / train['impressions'].sum()      # constant benchmark
def wrmse(y, p, w): return np.sqrt(np.average((y - p) ** 2, weights=w))
summary = pd.DataFrame({m: dict(MAE=wmae(y, test['pred'+m], w), RMSE=wrmse(y, test['pred'+m], w),
                                bias=np.average(test['pred'+m] - y, weights=w)) for m in 'KABC'}).T
summary.index = ['constant', 'A position', 'B position x archetype', 'C tree']
print(summary.round(6))

floor = np.average(np.sqrt(test['predA'] * (1 - test['predA']) / test['impressions']) * np.sqrt(2 / np.pi), weights=w)
print('approx. noise floor (MAE): %.6f' % floor)

test['zero_clicks'] = test['clicks'] == 0
for col in ['pb', 'zero_clicks']:
    print(test.groupby(col).apply(lambda d: pd.Series({'pages': len(d),
          'MAE_A': wmae(d['ctr'], d['predA'], d['impressions']),
          'MAE_B': wmae(d['ctr'], d['predB'], d['impressions'])}), include_groups=False).round(5))

# ranking: the Week-4 filter matches no test page, so compare on all test pages
print('eligible test pages:', elig.sum())
tops = {m: set(test.nlargest(20, 'miss'+m)['content_hash_id']) for m in 'ABC'}
print('overlap A&B:', len(tops['A'] & tops['B']), ' A&C:', len(tops['A'] & tops['C']), ' B&C:', len(tops['B'] & tops['C']))

from sklearn.inspection import permutation_importance
pi = permutation_importance(tree, XteC, test['ctr'], sample_weight=test['impressions'],
                            scoring='neg_mean_absolute_error', n_repeats=10, random_state=0)
print(pd.Series(pi.importances_mean, index=XteC.columns).sort_values(ascending=False).round(6))

                             MAE      RMSE      bias
constant                0.003584  0.006147 -0.001978
A position              0.003445  0.005943 -0.001714
B position x archetype  0.003533  0.006038 -0.001891
C tree                  0.003616  0.006129 -0.001839
approx. noise floor (MAE): 0.000905
            pages    MAE_A    MAE_B
pb                                 
(0, 3]      181.0  0.00490  0.00564
(10, 20]   5759.0  0.00367  0.00376
(20, 50]   4878.0  0.00234  0.00235
(3, 5]     1553.0  0.00461  0.00468
(5, 7]     3375.0  0.00336  0.00339
(50, 100]   748.0  0.00090  0.00090
(7, 10]    4983.0  0.00306  0.00313
               pages    MAE_A    MAE_B
zero_clicks                           
False        15184.0  0.00349  0.00359
True          6293.0  0.00281  0.00264
eligible test pages: 0
overlap A&B: 18  A&C: 13  B&C: 12
word_count             0.000010
search_volume          0.000003
category_count         0.000003
cpc                    0.000000
backlinks              0.000000
ke

In [38]:
print(pd.crosstab(train['cluster'], train['client'], normalize='index').max(axis=1).round(2))

cluster
0    0.19
1    0.23
2    0.71
3    0.18
4    0.22
dtype: float64


Cluster composition. Four of five clusters mix many clients (the largest client is 18% to 23% of each), but cluster 2, the long-article, poorly ranked group, is 71% one client, so it is better read as one client’s content style than as a general archetype. Its share falls from 27% of training pages to 9% of test pages, which changes the mix between train and test [confirm after the check above].

In [39]:
CORE = ['word_count','char_count','keyword_token_count','url_char_count','category_count']

def run_B(train, test, feats, K, min_n=50):
    Xtr_ = np.log1p(train[feats].clip(lower=0)); Xte_ = np.log1p(test[feats].clip(lower=0))
    sc_ = StandardScaler().fit(Xtr_)
    km_ = KMeans(K, n_init=10, random_state=42).fit(sc_.transform(Xtr_))
    tr = train.assign(cluster=km_.labels_)
    te = test.assign(cluster=km_.predict(sc_.transform(Xte_)))
    cell_ = tr.groupby(['pb','cluster']).agg(n=('ctr','size'), c=('clicks','sum'), i=('impressions','sum'))
    cell_['pred'] = cell_['c'] / cell_['i']
    te = te.join(cell_.loc[cell_['n'] >= min_n, 'pred'].rename('pB'), on=['pb','cluster'])
    return te['pB'].fillna(te['predA']).values

y, w = test['ctr'].values, test['impressions'].values
rows = []
for name, feats in {'all 9': CORE, 'core 5': CORE,
                    'always-available 3': ['keyword_token_count','url_char_count','category_count']}.items():
    for K in (2, 3, 4, 5, 6):
        pB = run_B(train, test, feats, K)
        rows.append(dict(features=name, K=K, MAE_B=wmae(y, pB, w),
                         B_minus_A=np.average(np.abs(y - pB) - np.abs(y - test['predA'].values), weights=w)))
print(pd.DataFrame(rows).round(6))

              features  K     MAE_B  B_minus_A
0                all 9  2  0.003526   0.000081
1                all 9  3  0.003535   0.000090
2                all 9  4  0.003447   0.000002
3                all 9  5  0.003536   0.000091
4                all 9  6  0.003519   0.000073
5               core 5  2  0.003526   0.000081
6               core 5  3  0.003535   0.000090
7               core 5  4  0.003447   0.000002
8               core 5  5  0.003536   0.000091
9               core 5  6  0.003519   0.000073
10  always-available 3  2  0.003453   0.000008
11  always-available 3  3  0.003509   0.000064
12  always-available 3  4  0.003488   0.000043
13  always-available 3  5  0.003493   0.000047
14  always-available 3  6  0.003494   0.000049


Sensitivity. Across 15 setups (three feature sets, K = 2 to 6), the archetype-aware benchmark never measured better than position alone: it was between 0.06% and 3.5% worse in MAE. The closest rows are essentially ties; the primary setting (all nine features, K=5) is not the most favourable one. This table has no confidence intervals, so small gaps should be read as “not distinguishable from the baseline”.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.